# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/peddikotlahimani/Flyrank-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule: A page is likely to have a good click-through rate if it gets a decent number of impressions and has a proper amount of content. If it gets enough impressions but people still aren't clicking, that's worth flagging as a signal problem.

good_ctr — the page is getting impressions and its CTR is at or above average.

low_ctr_but_visible — the page is getting enough impressions, but CTR is below average (can be flagged).

not_enough_visibility — the page isn't getting enough impressions to judge CTR meaningfully yet.

missing_content_data — word count (or other content info) is missing for this page, so the rule can't fully judge it.



In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

#make sure all needed columns exist
feature_vector["had_impressions"] = feature_vector["gsc_impressions"] > 0
feature_vector["ctr_1h"] = feature_vector["ctr_1h"].fillna(0)
feature_vector["gsc_avg_position"] = feature_vector["gsc_avg_position"].fillna(100)

#assign reason codes
median_impressions = feature_vector["gsc_impressions"].median()

def assign_reason_code(row):
    if row["had_impressions"] == False:
        return "no_impressions"
    elif row["gsc_impressions"] >= median_impressions and row["gsc_avg_position"] <= 10:
        return "good_visibility"
    else:
        return "poor_visibility"

feature_vector["reason_code"] = feature_vector.apply(assign_reason_code, axis=1)

#turn reason codes into a numeric score
score_map = {
    "good_visibility": 2,
    "poor_visibility": 1,
    "no_impressions": 0
}

feature_vector["baseline_score"] = feature_vector["reason_code"].map(score_map)

#sort all pages, best score first, and add a rank number
ranked_queue = feature_vector.sort_values("baseline_score", ascending=False).reset_index(drop=True)
ranked_queue["rank"] = ranked_queue.index + 1

#save it as a CSV file
import os
os.makedirs("work/outputs", exist_ok=True)
ranked_queue.to_csv("work/outputs/baseline_action_score.csv", index=False)

#confirm it saved correctly
print("Saved successfully!")
print("File exists:", os.path.exists("work/outputs/baseline_action_score.csv"))
ranked_queue.head(10)

Saved successfully!
File exists: True


,content_hash_id,client_hash_id,gsc_clicks,gsc_impressions,gsc_avg_position,ga4_engaged_sessions,ga4_sessions,ctr_1h,engagement_rate_1h,had_impressions,reason_code,baseline_score,rank
0,content_000bb404440dc5f1,client_20259bd6705d81d4,24,9528,3.813321,2,29,0.002519,0.068966,True,good_visibility,2,1
1,content_ffffc58385523096,client_e547b89c05043229,22,1145,3.882007,2,17,0.019214,0.117647,True,good_visibility,2,2
2,content_000eeecc6cd7de41,client_fef1a8f436438636,1,74,7.418919,0,1,0.013514,0.0,True,good_visibility,2,3
3,content_00032be2df0005ca,client_fef1a8f436438636,2,140,6.771232,0,42,0.014286,0.0,True,good_visibility,2,4
4,content_fff680fff14351d0,client_23a62021009f63c4,20,631,5.621621,3,32,0.031696,0.09375,True,good_visibility,2,5
5,content_b7480bdac3334021,client_20259bd6705d81d4,0,29,1.379310,0,1,0.000000,0.0,True,good_visibility,2,6
6,content_b7513c5dd5cda0d5,client_23a62021009f63c4,1,95,8.732786,0,4,0.010526,0.0,True,good_visibility,2,7
7,content_b7540122e8d3e612,client_fef1a8f436438636,1,77,6.381548,1,20,0.012987,0.05,True,good_visibility,2,8
8,content_b736513d8abbdc8b,client_3f0ce4d44fe94f3d,6,232,2.880501,0,5,0.025862,0.0,True,good_visibility,2,9
9,content_004047a006c4edc5,client_65de48885f4ef01b,3,76,4.414457,1,5,0.039474,0.2,True,good_visibility,2,10


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*



| # | Page | Impressions | Clicks before→after | % drop | Action | Confidence | What would make it wrong |
|---|---|---|---|---|---|---|---|
| 1 | ec2e03 | 245,276 | 843→637 | 24% | Review for refresh | Medium | Could be a seasonal dip |
| 2 | 0ec909 | 119,730 | 749→612 | 18% | Monitor only | Low | Normal noise, not real decline |
| 3 | c9a0c2 | 65,681 | 439→300 | 32% | Review for refresh | Medium | Competitor may have outranked it |
| 4 | e8a52c | 244,931 | 353→316 | 10% | Monitor only | Low | Likely normal fluctuation |
| 5 | 5ebc94 | 20,924 | 300→131 | 56% | Review for refresh | High | Sensitive to random noise (smaller audience) |
| 6 | 017587 | 15,276 | 284→51 | 82% | Review urgently | High | Check for technical/broken page issue |
| 7 | 9fff53 | 90,756 | 242→197 | 19% | Monitor only | Low | Probably normal variation |
| 8 | 19b310 | 36,389 | 242→189 | 22% | Review for refresh | Medium | Seasonal risk |
| 9 | 40baa8 | 59,311 | 237→209 | 12% | Monitor only | Low | Likely noise, not decline |
| 10 | 3f8597 | 75,204 | 236→162 | 31% | Review for refresh | Medium | Competitor ranking shift possible |
| 11 | ef471d | 38,319 | 232→151 | 35% | Review for refresh | Medium | — |
| 12 | 8d7d99 | 181,942 | 232→54 | 77% | Review urgently | High | Check for technical/indexing problem |
| 13 | 747ff6 | 30,796 | 214→89 | 58% | Review for refresh | High | — |
| 14 | ce467e | 19,103 | 199→55 | 72% | Review for refresh | High | Small-sample noise possible |
| 15 | 623f10 | 23,139 | 189→73 | 61% | Review for refresh | High | Small-sample noise possible |
| 16 | 2e1a34 | 61,351 | 183→102 | 44% | Review for refresh | Medium | — |
| 17 | b5bef9 | 14,738 | 172→143 | 17% | Monitor only | Low | Likely just noise |
| 18 | 1f6c41 | 13,161 | 167→141 | 16% | Monitor only | Low | Might not be a real decline |
| 19 | 629b2d | 59,198 | 166→140 | 16% | Monitor only | Low | Could recover on its own |
| 20 | bc6c40 | 43,322 | 150→124 | 17% | Monitor only | Low | Could recover on its own |

In [ ]:
from google.colab import userdata
import duckdb

#open a connection to database
con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")

#access the data
my_token = userdata.get("HF_TOKEN")
con.sql("CREATE SECRET hf_token (TYPE huggingface, TOKEN '" + my_token + "');")
print("Connected!")

#question we want to ask the data
my_question = """
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_clicks) AS total_clicks,
        SUM(gsc_impressions) AS total_impressions,
        AVG(gsc_avg_position) AS avg_position,
        SUM(CASE WHEN report_date >= '2026-03-16' THEN gsc_clicks ELSE 0 END) AS clicks_2nd_half,
        SUM(CASE WHEN report_date < '2026-03-16' THEN gsc_clicks ELSE 0 END) AS clicks_1st_half
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    WHERE ga4_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
"""
#run the question and save the answer as "monthly"
monthly = con.sql(my_question).df()

#check it worked
print("It worked! Number of rows:", len(monthly))
#give each page a score
monthly["score"] = 0   #start everyone at 0

is_visible = monthly["total_impressions"] >= 50
had_traffic_before = monthly["clicks_1st_half"] > 0
went_down = monthly["clicks_2nd_half"] < monthly["clicks_1st_half"]

# only pages that are visible AND had traffic AND went down get a real score
should_flag = is_visible & had_traffic_before & went_down
monthly.loc[should_flag, "score"] = monthly.loc[should_flag, "clicks_1st_half"]

print("Done adding scores!")
def figure_out_reason(row):
    if row["total_impressions"] < 50:
        return "not_visible"
    elif row["clicks_1st_half"] == 0:
        return "no_prior_traffic"
    elif row["clicks_2nd_half"] < row["clicks_1st_half"]:
        return "declining_with_visibility"
    else:
        return "stable_or_growing"

monthly["reason_code"] = monthly.apply(figure_out_reason, axis=1)
print(monthly["reason_code"].value_counts())

Connected!


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

It worked! Number of rows: 90489
Done adding scores!
reason_code
not_visible                  51640
no_prior_traffic             20547
stable_or_growing            10130
declining_with_visibility     8172
Name: count, dtype: int64


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

sorted_data = monthly.sort_values("score", ascending=False)
top_20 = sorted_data.head(20)
top_20[["content_hash_id", "score", "reason_code", "total_impressions", "clicks_1st_half", "clicks_2nd_half"]]


,content_hash_id,score,reason_code,total_impressions,clicks_1st_half,clicks_2nd_half
47416,content_ec2e0346994fb5a5,843,declining_with_visibility,245276.0,843.0,637.0
6755,content_0ec90963d98b97a5,749,declining_with_visibility,119730.0,749.0,612.0
47267,content_c9a0c2fdbdbfb562,439,declining_with_visibility,65681.0,439.0,300.0
50512,content_e8a52cf3d5988c07,353,declining_with_visibility,244931.0,353.0,316.0
564,content_5ebc94f67db6f51c,300,declining_with_visibility,20924.0,300.0,131.0
6916,content_0175875757a5a1b3,284,declining_with_visibility,15276.0,284.0,51.0
52032,content_9fff53e827550f9d,242,declining_with_visibility,90756.0,242.0,197.0
5015,content_19b310673809a9ee,242,declining_with_visibility,36389.0,242.0,189.0
707,content_40baa8f1016f5742,237,declining_with_visibility,59311.0,237.0,209.0
6721,content_3f8597ccc4b874a9,236,declining_with_visibility,75204.0,236.0,162.0


## 4. Weak picks + leakage check
*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak Picks: 8 of my top 20 show less than a 20% click drop despite scoring high: content_0ec90963d98b97a5 (18%), content_e8a52cf3d5988c07 (10%), content_9fff53e827550f9d (19%), content_40baa8f1016f5742 (12%), content_b5bef91d1b43e20c (17%), content_1f6c41e4396331c5 (16%), content_629b2d2f28c32b39 (16%), content_bc6c40f278e1b785 (17%). These picks look wrong because my scoring formula uses the raw number of clicks lost, not the percentage decline — so a huge page with a tiny (10-17%) drop scores higher than it deserves, simply because it started with more clicks to lose, not because it's genuinely declining.



In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(monthly.columns.tolist())

date_check = con.sql("""
    SELECT MIN(report_date) AS earliest_date, MAX(report_date) AS latest_date
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
""").df()
print(date_check)


['content_hash_id', 'client_hash_id', 'total_clicks', 'total_impressions', 'avg_position', 'clicks_2nd_half', 'clicks_1st_half', 'score', 'reason_code']
  earliest_date latest_date
0    2026-03-01  2026-03-31


## Self-check

Before you submit, confirm each line honestly:

- [yes ] Every section above is filled — markdown thinking AND the code that backs it
- [yes ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [yes ] No client names, URLs, or private queries anywhere
- [yes ] My claims use careful words: observed, measured, directional, decision-support
- [yes ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.